# 🏨 Actividad 09 — Servidor MCP de Búsqueda de Hoteles

## ¿Qué construirás?

Un **servidor MCP** (Model Context Protocol de Anthropic) con 6 herramientas de búsqueda
real de hoteles y destinos, más un **cliente Gradio** con tres agentes especializados
con acceso controlado según su perfil geográfico.

| Agente | Herramientas | Alcance |
|--------|-------------|---------|
| 🇪🇸 España | `buscar_hoteles_espana`, `buscar_destinos_espana` | Solo España |
| 🌍 Europa | `buscar_hoteles_europa`, `buscar_destinos_europa` | Cualquier país europeo |
| 🌐 Global | `buscar_hoteles_global`, `buscar_destinos_global` | Sin restricción |

## Arquitectura

```
  Gradio UI (3 pestañas)
       │ control de acceso por agente
       ▼
  MCP Client (Python)
       │ HTTP / SSE
       ▼
  MCP Server — proceso independiente, puerto 8000
  ├── buscar_hoteles_espana   ──┐
  ├── buscar_destinos_espana  ──┤
  ├── buscar_hoteles_europa   ──┤── DuckDuckGo
  ├── buscar_destinos_europa  ──┤
  ├── buscar_hoteles_global   ──┤
  └── buscar_destinos_global  ──┘
```

## ¿Por qué MCP en lugar de herramientas LangChain?

| Herramientas en código (Act. 08) | Herramientas MCP (esta actividad) |
|----------------------------------|-----------------------------------|
| Acopladas al agente | Servidor independiente y reutilizable |
| Sin contrato formal | JSON Schema define entradas/salidas |
| Un solo cliente | Múltiples clientes con acceso controlado |


In [ ]:
# ============================================================
# PARTE 1 — Instalación de dependencias
# ============================================================

# mcp: paquete oficial de Anthropic para construir servidores y clientes MCP
# duckduckgo-search: búsqueda web gratuita sin clave de API
# gradio: interfaz web interactiva
# pyngrok: publicar la interfaz fuera de Colab
# nest-asyncio: permite asyncio.run() dentro del event loop de Colab/IPython
# uvicorn: servidor ASGI que FastMCP usa internamente para el transporte SSE
!pip install -q mcp duckduckgo-search gradio pyngrok nest-asyncio uvicorn

print('✅ Dependencias instaladas correctamente')


## 📡 PARTE 2 — ¿Qué es el Model Context Protocol (MCP)?

MCP es un **protocolo abierto de Anthropic** que estandariza cómo los sistemas de IA
se comunican con herramientas externas y fuentes de datos. Es el "USB del mundo de la IA":
cualquier cliente MCP puede conectarse a cualquier servidor MCP.

### Diagrama del protocolo

```
┌─────────────────┐     protocolo MCP      ┌──────────────────────┐
│   MCP Client    │ ◄──────────────────►   │    MCP Server        │
│ (tu código,     │                        │ (proceso separado)   │
│  Gradio, etc.)  │  1. list_tools()       │                      │
│                 │  2. call_tool(name,    │  Tools  (funciones)  │
│                 │     arguments)         │  Resources (datos)   │
│                 │  ◄── resultado         │  Prompts (plantillas)│
└─────────────────┘                        └──────────────────────┘

  Transportes disponibles:
  • stdio    — comunicación por stdin/stdout (subproceso hijo)
  • SSE      — HTTP con Server-Sent Events  ← usamos este en Colab
  • WebSocket — conexión bidireccional persistente
```

### ¿Por qué SSE en Colab y no stdio?

`stdio` requiere que el cliente controle el stdin/stdout del servidor. En Colab, el
notebook ya ocupa esos canales, lo que genera conflictos. **SSE sobre HTTP** es independiente:
el servidor corre en su propio proceso en el puerto 8000 y el cliente se conecta por HTTP.

### Los tres tipos de capacidades MCP

| Tipo | Descripción | En esta actividad |
|------|-------------|-------------------|
| **Tools** | Funciones que el cliente puede llamar | 6 tools de búsqueda |
| **Resources** | Datos o archivos que el servidor expone | No se usan |
| **Prompts** | Plantillas de mensajes reutilizables | No se usan |


In [ ]:
# ============================================================
# CELDA 2.1 — Verificar importaciones del paquete MCP
# ============================================================

import mcp                              # Paquete principal de Model Context Protocol
from mcp.server.fastmcp import FastMCP  # API de alto nivel: define tools con decoradores
from mcp import ClientSession           # Gestiona la sesión de cliente MCP
from mcp.client.sse import sse_client   # Conecta al servidor por HTTP/SSE

import nest_asyncio                     # Permite anidar event loops en Colab
nest_asyncio.apply()                    # Parchea el event loop; solo se llama una vez

import asyncio                          # Módulo estándar para programación asíncrona

print(f'✅ MCP versión: {mcp.__version__}')
print('✅ FastMCP, ClientSession, sse_client importados')
print('✅ nest_asyncio aplicado — asyncio.run() funcionará en Colab')


## 🛠️ PARTE 3 — Escribir el Servidor MCP

El servidor se escribe en un archivo `.py` separado porque correrá como **proceso
independiente**. Esta separación es la esencia de MCP: el servidor no sabe nada del
cliente; podría ser consumido desde Claude Desktop, VS Code o cualquier otro cliente MCP.

### ¿Cómo define FastMCP una tool?

```python
@mcp.tool()
def mi_herramienta(ciudad: str, presupuesto: int) -> str:
    '''Descripción visible por el cliente en list_tools().'''
    return f'resultado para {ciudad}'
```

FastMCP hace dos cosas con ese decorador:
1. **Registra** la función como tool MCP con su nombre y descripción
2. **Genera el JSON Schema** de parámetros a partir de los type hints de Python

### Las 6 herramientas de este servidor

| Tool | Parámetros | Alcance |
|------|-----------|--------|
| `buscar_hoteles_espana` | `ciudad`, `presupuesto` | Solo España |
| `buscar_destinos_espana` | `tipo` | Solo España |
| `buscar_hoteles_europa` | `ciudad`, `pais`, `presupuesto` | Europa |
| `buscar_destinos_europa` | `pais`, `tipo` | Europa |
| `buscar_hoteles_global` | `ciudad`, `pais`, `presupuesto` | Mundial |
| `buscar_destinos_global` | `region`, `tipo` | Mundial |


In [ ]:
%%writefile mcp_server.py
# ============================================================
# mcp_server.py — Servidor MCP de búsqueda de hoteles
# ============================================================
# Este archivo corre como proceso independiente.
# El cliente se conecta por HTTP usando el transporte SSE.

from mcp.server.fastmcp import FastMCP   # API de alto nivel de Anthropic para MCP
from duckduckgo_search import DDGS       # Motor de búsqueda gratuito, sin API key

# Creamos el servidor con un nombre identificador.
# Ese nombre aparece en los logs y en el handshake de inicialización del cliente.
mcp = FastMCP('hotel-search-server')


# ── HERRAMIENTAS DE ESPAÑA ─────────────────────────────────

@mcp.tool()
def buscar_hoteles_espana(ciudad: str, presupuesto: int) -> str:
    '''Busca hoteles en España dada una ciudad y presupuesto máximo por noche en euros.'''
    query = f'hoteles {ciudad} España precio noche hasta {presupuesto} euros valoracion'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))  # Máx 5 para no saturar
    if not resultados:
        return f'No se encontraron hoteles en {ciudad} con presupuesto de {presupuesto}€.'
    salida = f'🏨 Hoteles en {ciudad}, España — hasta {presupuesto}€/noche:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']   # Título del resultado de búsqueda
        cuerpo = r['body']    # Fragmento descriptivo del resultado
        url    = r['href']    # URL del resultado
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


@mcp.tool()
def buscar_destinos_espana(tipo: str) -> str:
    '''Busca destinos turísticos en España por tipo: playa, montaña, ciudad, rural, cultural.'''
    query = f'mejores destinos {tipo} España turismo 2024 que ver'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))
    if not resultados:
        return f'No se encontraron destinos de tipo {tipo} en España.'
    salida = f'🗺️ Destinos {tipo} en España:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']
        cuerpo = r['body']
        url    = r['href']
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


# ── HERRAMIENTAS DE EUROPA ─────────────────────────────────

@mcp.tool()
def buscar_hoteles_europa(ciudad: str, pais: str, presupuesto: int) -> str:
    '''Busca hoteles en Europa dada una ciudad, país europeo y presupuesto máximo por noche.'''
    query = f'hoteles {ciudad} {pais} Europa precio noche {presupuesto} euros'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))
    if not resultados:
        return f'No se encontraron hoteles en {ciudad}, {pais}.'
    salida = f'🏨 Hoteles en {ciudad}, {pais} — hasta {presupuesto}€/noche:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']
        cuerpo = r['body']
        url    = r['href']
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


@mcp.tool()
def buscar_destinos_europa(pais: str, tipo: str) -> str:
    '''Busca destinos turísticos en un país europeo por tipo de viaje.'''
    query = f'mejores destinos {tipo} {pais} Europa turismo 2024'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))
    if not resultados:
        return f'No se encontraron destinos de tipo {tipo} en {pais}.'
    salida = f'🗺️ Destinos {tipo} en {pais}, Europa:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']
        cuerpo = r['body']
        url    = r['href']
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


# ── HERRAMIENTAS GLOBALES ──────────────────────────────────

@mcp.tool()
def buscar_hoteles_global(ciudad: str, pais: str, presupuesto: int) -> str:
    '''Busca hoteles en cualquier parte del mundo dada ciudad, país y presupuesto.'''
    query = f'hotels {ciudad} {pais} price per night {presupuesto} EUR booking review'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))
    if not resultados:
        return f'No se encontraron hoteles en {ciudad}, {pais}.'
    salida = f'🌐 Hoteles en {ciudad}, {pais} — hasta {presupuesto}€:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']
        cuerpo = r['body']
        url    = r['href']
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


@mcp.tool()
def buscar_destinos_global(region: str, tipo: str) -> str:
    '''Busca los mejores destinos turísticos en cualquier región del mundo.'''
    query = f'best {tipo} destinations {region} 2024 travel tourism'
    with DDGS() as ddgs:
        resultados = list(ddgs.text(query, max_results=5))
    if not resultados:
        return f'No se encontraron destinos de tipo {tipo} en {region}.'
    salida = f'🌍 Destinos {tipo} en {region}:\n\n'
    for i, r in enumerate(resultados, 1):
        titulo = r['title']
        cuerpo = r['body']
        url    = r['href']
        salida += f'{i}. **{titulo}**\n{cuerpo}\n🔗 {url}\n\n'
    return salida


# Arranca el servidor con transporte SSE en el puerto 8000.
# El cliente se conectará a http://localhost:8000/sse
if __name__ == '__main__':
    mcp.run(transport='sse', host='0.0.0.0', port=8000)


## 🚀 PARTE 4 — Arrancar el Servidor MCP como Proceso Independiente

El servidor MCP se ejecuta como un **subproceso** separado del notebook.
Esta separación replica exactamente cómo funcionaría en producción:
el servidor corre en su propia memoria y el cliente se comunica con él por HTTP.

```
Notebook (proceso principal)
     │
     ├── subprocess.Popen → inicia mcp_server.py en background
     │                           │
     │                           └── FastMCP escucha en :8000
     │
     └── MCP Client ──HTTP/SSE──► http://localhost:8000/sse
```

> **Importante:** si reinicias el kernel debes volver a ejecutar esta celda.
> El proceso hijo no sobrevive al reinicio del kernel de Colab.


In [ ]:
# ============================================================
# CELDA 4.1 — Arrancar el servidor MCP en background
# ============================================================

import subprocess  # Para lanzar el servidor como proceso independiente

# Popen lanza el proceso sin bloquear el notebook.
# El servidor queda corriendo en segundo plano mientras ejecutamos el resto.
servidor_proceso = subprocess.Popen(
    ['python', 'mcp_server.py'],  # Ejecuta el archivo que escribimos en la Parte 3
    stdout=subprocess.PIPE,       # Captura la salida estándar (logs internos del servidor)
    stderr=subprocess.PIPE        # Captura los errores para mostrarlos si el servidor falla
)

print(f'✅ Proceso del servidor MCP iniciado (PID: {servidor_proceso.pid})')
print('   Esperando que el servidor arranque en http://localhost:8000...')


In [ ]:
# ============================================================
# CELDA 4.2 — Esperar a que el servidor esté listo
# ============================================================

import socket  # Para verificar si el puerto 8000 está escuchando conexiones
import time    # Para las pausas entre intentos

print('⏳ Verificando disponibilidad del servidor MCP...')

# Intentamos conectarnos al puerto 8000 hasta 30 veces (30 segundos máximo).
# El servidor tarda típicamente 2-4 segundos en arrancar uvicorn.
for intento in range(30):
    try:
        # create_connection lanza ConnectionRefusedError si el puerto no está abierto
        conexion = socket.create_connection(('localhost', 8000), timeout=1)
        conexion.close()  # Cerramos inmediatamente; solo queríamos verificar que responde
        print(f'✅ Servidor MCP listo en http://localhost:8000 (tardó {intento + 1}s)')
        break
    except (socket.timeout, ConnectionRefusedError):
        time.sleep(1)  # Esperamos 1 segundo antes del siguiente intento
        if (intento + 1) % 5 == 0:
            print(f'   Esperando... {intento + 1}s')
else:
    # Si llegamos aquí, el servidor nunca respondió en 30 segundos
    log_error = servidor_proceso.stderr.read(500).decode('utf-8', errors='replace')
    print(f'❌ El servidor no arrancó en 30 segundos.')
    print(f'   Últimos errores capturados:\n{log_error}')


## 🧪 PARTE 5 — Probar las Herramientas del Servidor

Antes de construir la interfaz Gradio, probamos **cada herramienta por separado**
directamente contra el servidor MCP. Esto nos permite verificar que:

1. El servidor acepta conexiones de clientes MCP
2. Las tools están registradas con sus parámetros correctos
3. DuckDuckGo devuelve resultados reales de búsqueda

Usamos `asyncio.run()` directamente porque ya aplicamos `nest_asyncio` en la Parte 2.


In [ ]:
# ============================================================
# CELDA 5.1 — Listar todas las herramientas registradas en el servidor
# ============================================================

async def listar_herramientas():
    '''Conecta al servidor MCP y devuelve la lista completa de tools disponibles.'''
    # sse_client abre la conexión SSE al servidor como context manager asíncrono
    async with sse_client('http://localhost:8000/sse') as (canal_r, canal_w):
        # ClientSession gestiona el protocolo: handshake, mensajes y cierre
        async with ClientSession(canal_r, canal_w) as sesion:
            await sesion.initialize()           # Handshake inicial: intercambia capacidades
            respuesta = await sesion.list_tools()  # Solicita la lista de tools registradas
            return respuesta.tools

# asyncio.run() funciona aquí porque nest_asyncio está aplicado
herramientas = asyncio.run(listar_herramientas())

print(f'✅ El servidor tiene {len(herramientas)} herramientas registradas:\n')
for tool in herramientas:
    print(f'  🔧 {tool.name}')
    print(f'     {tool.description}')
    # Mostramos los parámetros que acepta la tool según su JSON Schema
    if tool.inputSchema and 'properties' in tool.inputSchema:
        params = list(tool.inputSchema['properties'].keys())
        print(f'     Parámetros: {params}')
    print()


In [ ]:
# ============================================================
# CELDA 5.2 — Test: buscar_hoteles_espana
# ============================================================

# Función auxiliar de prueba para llamar cualquier tool por nombre
async def llamar_tool_test(nombre_tool: str, argumentos: dict) -> str:
    '''Llama una tool del servidor MCP y devuelve el texto del resultado.'''
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(leer, escribir) as sesion:
            await sesion.initialize()
            resultado = await sesion.call_tool(nombre_tool, argumentos)
            # resultado.content es una lista de ContentBlocks; tomamos el primer texto
            return resultado.content[0].text if resultado.content else 'Sin resultado'

# 🔧 PARÁMETRO: cambia la ciudad o el presupuesto para probar otros valores
respuesta = asyncio.run(llamar_tool_test(
    'buscar_hoteles_espana',
    {'ciudad': 'Barcelona', 'presupuesto': 100}
))

print('=== TEST: buscar_hoteles_espana(ciudad=Barcelona, presupuesto=100) ===\n')
print(respuesta[:1500])  # Limitamos a 1500 caracteres para no saturar la salida
print('\n✅ Tool buscar_hoteles_espana funcionando correctamente')


In [ ]:
# ============================================================
# CELDA 5.3 — Test: buscar_destinos_espana
# ============================================================

# 🔧 PARÁMETRO: prueba con 'montaña', 'ciudad', 'rural' o 'cultural'
respuesta = asyncio.run(llamar_tool_test(
    'buscar_destinos_espana',
    {'tipo': 'playa'}
))

print('=== TEST: buscar_destinos_espana(tipo=playa) ===\n')
print(respuesta[:1500])
print('\n✅ Tool buscar_destinos_espana funcionando correctamente')


In [ ]:
# ============================================================
# CELDA 5.4 — Test: buscar_hoteles_europa
# ============================================================

respuesta = asyncio.run(llamar_tool_test(
    'buscar_hoteles_europa',
    {'ciudad': 'París', 'pais': 'Francia', 'presupuesto': 150}
))

print('=== TEST: buscar_hoteles_europa(ciudad=París, pais=Francia, presupuesto=150) ===\n')
print(respuesta[:1500])
print('\n✅ Tool buscar_hoteles_europa funcionando correctamente')


In [ ]:
# ============================================================
# CELDA 5.5 — Test: buscar_hoteles_global
# ============================================================

respuesta = asyncio.run(llamar_tool_test(
    'buscar_hoteles_global',
    {'ciudad': 'Tokio', 'pais': 'Japón', 'presupuesto': 200}
))

print('=== TEST: buscar_hoteles_global(ciudad=Tokio, pais=Japón, presupuesto=200) ===\n')
print(respuesta[:1500])
print('\n✅ Las 6 tools del servidor están operativas y devuelven resultados reales')


## 🔐 PARTE 6 — Cliente MCP con Control de Acceso por Agente

Hasta ahora hemos llamado cualquier tool libremente. Ahora añadimos la capa de
**control de acceso**: cada agente solo puede usar las tools asignadas a su perfil.

### ¿Cómo funciona?

```python
AGENT_TOOLS = {
    '🇪🇸 España': ['buscar_hoteles_espana', 'buscar_destinos_espana'],
    '🌍 Europa':  ['buscar_hoteles_europa',  'buscar_destinos_europa'],
    '🌐 Global':  ['buscar_hoteles_global',  'buscar_destinos_global'],
}
```

Antes de cada llamada, `ejecutar_con_control()` verifica si el agente tiene permiso.
Si no lo tiene, devuelve un error **sin llegar al servidor**.

```
Agente España pide: buscar_hoteles_europa
     │
     ▼
ejecutar_con_control()
     │ ¿'buscar_hoteles_europa' en lista de España?
     │ NO → ❌ Acceso denegado (no llega al servidor)
     │ SÍ → ✅ Llama al servidor MCP
     ▼
MCP Server
```

> **Nota de diseño:** El control de acceso está en el cliente, no en el servidor.
> El servidor no sabe nada de las restricciones: cualquier cliente podría llamar
> cualquier tool directamente. Esta arquitectura es válida para demostración;
> en producción el control debería estar en ambos lados.


In [ ]:
# ============================================================
# CELDA 6.1 — Definir el cliente MCP con control de acceso
# ============================================================

# ── Función base: llama una tool del servidor MCP ──────────

async def llamar_tool(nombre_tool: str, argumentos: dict) -> str:
    '''Conecta al servidor MCP, ejecuta la tool y devuelve el resultado como texto.'''
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(leer, escribir) as sesion:
            await sesion.initialize()
            resultado = await sesion.call_tool(nombre_tool, argumentos)
            return resultado.content[0].text if resultado.content else 'Sin resultado'

# ── Tabla de acceso: qué tools puede usar cada agente ──────

# 🔧 PARÁMETRO: modifica las listas para cambiar el control de acceso de cada agente
AGENT_TOOLS = {
    '🇪🇸 España':  ['buscar_hoteles_espana', 'buscar_destinos_espana'],
    '🌍 Europa':   ['buscar_hoteles_europa',  'buscar_destinos_europa'],
    '🌐 Global':   ['buscar_hoteles_global',  'buscar_destinos_global'],
}

# ── Función con control: verifica permiso antes de llamar ──

def ejecutar_con_control(agente: str, nombre_tool: str, argumentos: dict) -> str:
    '''Verifica que el agente tenga acceso a la tool; si no, retorna mensaje de error.'''
    tools_permitidas = AGENT_TOOLS.get(agente, [])  # Lista de tools del agente

    if nombre_tool not in tools_permitidas:
        # El agente no tiene permiso; no llegamos al servidor MCP
        return (
            f'❌ **Acceso denegado**\n\n'
            f'El agente `{agente}` no puede usar `{nombre_tool}`.\n\n'
            f'Tools permitidas para este agente: `{tools_permitidas}`'
        )

    # El agente tiene permiso: llamamos al servidor MCP de forma síncrona
    return asyncio.run(llamar_tool(nombre_tool, argumentos))

print('✅ AGENT_TOOLS configurado:')
for agente, tools in AGENT_TOOLS.items():
    print(f'  {agente}: {tools}')
print('\n✅ Funciones llamar_tool y ejecutar_con_control listas')


## 🧪 PARTE 7 — Verificar el Control de Acceso

Tres pruebas para confirmar que los agentes respetan sus límites:

| Prueba | Agente | Tool solicitada | Resultado esperado |
|--------|--------|-----------------|--------------------|
| 7.1 | 🇪🇸 España | `buscar_hoteles_espana` | ✅ Devuelve resultados |
| 7.2 | 🇪🇸 España | `buscar_hoteles_europa` | ❌ Acceso denegado |
| 7.3 | 🌍 Europa | `buscar_destinos_europa` | ✅ Devuelve resultados |


In [ ]:
# ============================================================
# CELDA 7.1 — Agente España llama una tool PERMITIDA
# ============================================================

resultado = ejecutar_con_control(
    '🇪🇸 España',            # Agente que hace la solicitud
    'buscar_hoteles_espana', # Tool que solicita (está en su lista)
    {'ciudad': 'Sevilla', 'presupuesto': 80}
)

print('=== PRUEBA 7.1: España → buscar_hoteles_espana (DEBE FUNCIONAR) ===\n')
print(resultado[:1000])
print('\n✅ El agente España puede usar sus tools correctamente')


In [ ]:
# ============================================================
# CELDA 7.2 — Agente España intenta una tool DENEGADA
# ============================================================

# El agente España intenta usar una tool de Europa.
# El control de acceso debe bloquearlo SIN llegar al servidor.
resultado = ejecutar_con_control(
    '🇪🇸 España',            # Agente con restricción geográfica
    'buscar_hoteles_europa', # Esta tool NO está en su lista de acceso
    {'ciudad': 'Roma', 'pais': 'Italia', 'presupuesto': 120}
)

print('=== PRUEBA 7.2: España → buscar_hoteles_europa (DEBE SER DENEGADO) ===\n')
print(resultado)
print('\n✅ Control de acceso funciona: España no puede usar tools de Europa')


In [ ]:
# ============================================================
# CELDA 7.3 — Agente Europa llama una tool PERMITIDA
# ============================================================

resultado = ejecutar_con_control(
    '🌍 Europa',
    'buscar_destinos_europa',
    {'pais': 'Italia', 'tipo': 'ciudad'}
)

print('=== PRUEBA 7.3: Europa → buscar_destinos_europa (DEBE FUNCIONAR) ===\n')
print(resultado[:1000])
print('\n✅ Control de acceso verificado para los tres agentes')


## 🎨 PARTE 8 — Interfaz Gradio con Tres Agentes

La interfaz tiene **una pestaña por agente**. Cada pestaña:
- Muestra qué tools tiene disponibles (✅) y cuáles no (❌)
- Ofrece dos secciones: buscar hoteles y buscar destinos
- Usa `ejecutar_con_control()` para todas las llamadas — la restricción es automática

La interfaz se lanza en `localhost:7860`. En la Parte 11 la publicaremos con Ngrok.


In [ ]:
# ============================================================
# CELDA 8.1 — Interfaz Gradio con control de acceso por agente
# ============================================================

import gradio as gr  # Framework para interfaces web interactivas

# ── Funciones callback para cada agente y tipo de búsqueda ─
# Cada función llama a ejecutar_con_control con el agente y tool correctos.

def hoteles_espana(ciudad, presupuesto):
    return ejecutar_con_control('🇪🇸 España', 'buscar_hoteles_espana',
                                {'ciudad': ciudad, 'presupuesto': int(presupuesto)})

def destinos_espana(tipo):
    return ejecutar_con_control('🇪🇸 España', 'buscar_destinos_espana', {'tipo': tipo})

def hoteles_europa(ciudad, pais, presupuesto):
    return ejecutar_con_control('🌍 Europa', 'buscar_hoteles_europa',
                                {'ciudad': ciudad, 'pais': pais, 'presupuesto': int(presupuesto)})

def destinos_europa(pais, tipo):
    return ejecutar_con_control('🌍 Europa', 'buscar_destinos_europa', {'pais': pais, 'tipo': tipo})

def hoteles_global(ciudad, pais, presupuesto):
    return ejecutar_con_control('🌐 Global', 'buscar_hoteles_global',
                                {'ciudad': ciudad, 'pais': pais, 'presupuesto': int(presupuesto)})

def destinos_global(region, tipo):
    return ejecutar_con_control('🌐 Global', 'buscar_destinos_global', {'region': region, 'tipo': tipo})

# ── Construcción de la interfaz con gr.Blocks ──────────────
# gr.Blocks permite diseño libre; gr.Tabs agrupa las pestañas por agente.

with gr.Blocks(title='🏨 MCP Hotel Search', theme=gr.themes.Soft()) as demo:

    gr.Markdown('# 🏨 MCP Hotel Search — Control de Acceso por Agente MCP')
    gr.Markdown('Cada pestaña es un agente con acceso restringido a herramientas MCP diferentes.')

    with gr.Tabs():

        # ── AGENTE ESPAÑA ──────────────────────────────────
        with gr.Tab('🇪🇸 Agente España'):
            gr.Markdown('**Herramientas:** `buscar_hoteles_espana` ✅ · `buscar_destinos_espana` ✅ · tools Europa ❌ · tools Global ❌')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### 🏨 Buscar hoteles')
                    es_ciudad  = gr.Textbox(label='Ciudad', placeholder='Barcelona, Madrid, Sevilla...')
                    es_presup  = gr.Slider(20, 300, value=80, step=10, label='Presupuesto máximo (€/noche)')
                    es_h_btn   = gr.Button('Buscar hoteles en España', variant='primary')
                with gr.Column():
                    gr.Markdown('### 🗺️ Buscar destinos')
                    es_tipo  = gr.Dropdown(['playa', 'montaña', 'ciudad', 'rural', 'cultural'],
                                           label='Tipo de destino', value='playa')
                    es_d_btn = gr.Button('Buscar destinos en España', variant='primary')
            es_out = gr.Markdown(label='Resultados')
            es_h_btn.click(hoteles_espana, [es_ciudad, es_presup], es_out)
            es_d_btn.click(destinos_espana, [es_tipo], es_out)

        # ── AGENTE EUROPA ──────────────────────────────────
        with gr.Tab('🌍 Agente Europa'):
            gr.Markdown('**Herramientas:** tools España ❌ · `buscar_hoteles_europa` ✅ · `buscar_destinos_europa` ✅ · tools Global ❌')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### 🏨 Buscar hoteles')
                    eu_ciudad  = gr.Textbox(label='Ciudad', placeholder='París, Roma, Lisboa...')
                    eu_pais    = gr.Textbox(label='País', placeholder='Francia, Italia, Portugal...')
                    eu_presup  = gr.Slider(20, 400, value=120, step=10, label='Presupuesto máximo (€/noche)')
                    eu_h_btn   = gr.Button('Buscar hoteles en Europa', variant='primary')
                with gr.Column():
                    gr.Markdown('### 🗺️ Buscar destinos')
                    eu_pais_d = gr.Textbox(label='País europeo', placeholder='Italia, Grecia, Francia...')
                    eu_tipo   = gr.Dropdown(['playa', 'montaña', 'ciudad', 'rural', 'cultural'],
                                            label='Tipo de destino', value='ciudad')
                    eu_d_btn  = gr.Button('Buscar destinos en Europa', variant='primary')
            eu_out = gr.Markdown(label='Resultados')
            eu_h_btn.click(hoteles_europa, [eu_ciudad, eu_pais, eu_presup], eu_out)
            eu_d_btn.click(destinos_europa, [eu_pais_d, eu_tipo], eu_out)

        # ── AGENTE GLOBAL ──────────────────────────────────
        with gr.Tab('🌐 Agente Global'):
            gr.Markdown('**Herramientas:** `buscar_hoteles_global` ✅ · `buscar_destinos_global` ✅ · Sin restricción geográfica')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### 🏨 Buscar hoteles')
                    gl_ciudad = gr.Textbox(label='Ciudad', placeholder='Tokio, Nueva York, Dubái...')
                    gl_pais   = gr.Textbox(label='País', placeholder='Japón, EE.UU., Emiratos...')
                    gl_presup = gr.Slider(20, 500, value=150, step=10, label='Presupuesto máximo (€/noche)')
                    gl_h_btn  = gr.Button('Buscar hoteles (Global)', variant='primary')
                with gr.Column():
                    gr.Markdown('### 🗺️ Buscar destinos')
                    gl_region = gr.Textbox(label='Región / Continente', placeholder='Asia, América del Sur, África...')
                    gl_tipo   = gr.Dropdown(['beach', 'mountain', 'city', 'adventure', 'luxury'],
                                            label='Tipo de destino', value='city')
                    gl_d_btn  = gr.Button('Buscar destinos (Global)', variant='primary')
            gl_out = gr.Markdown(label='Resultados')
            gl_h_btn.click(hoteles_global, [gl_ciudad, gl_pais, gl_presup], gl_out)
            gl_d_btn.click(destinos_global, [gl_region, gl_tipo], gl_out)

# Lanzamos localmente; en la Parte 11 añadiremos Ngrok para hacerlo público
demo.launch(server_name='0.0.0.0', server_port=7860, share=False)
print('✅ Interfaz Gradio lanzada en http://localhost:7860')


## 💬 PARTE 9 — Reflexión

Responde las siguientes preguntas editando esta celda.

---

### Pregunta 1 — Conceptual
**¿Cuál es la diferencia principal entre las herramientas de LangChain (Actividad 08) y las tools de MCP (esta actividad)?**

> _Escribe tu respuesta aquí_

---

### Pregunta 2 — Protocolo
**¿Por qué se usa el transporte SSE en lugar de stdio para esta actividad en Colab? ¿Qué problema resuelve?**

> _Escribe tu respuesta aquí_

---

### Pregunta 3 — Seguridad
**El control de acceso está implementado en el cliente, no en el servidor. ¿Qué ventajas y riesgos tiene esta decisión de diseño? ¿Cómo se haría más seguro?**

> _Escribe tu respuesta aquí_

---

### Pregunta 4 — Aplicaciones reales
**Menciona un caso de uso real (fuera del dominio de hoteles) donde el patrón de múltiples agentes con acceso diferenciado a las mismas tools tendría valor. Justifica por qué.**

> _Escribe tu respuesta aquí_


## 🚀 PARTE 10 — Retos Opcionales

### Reto 1 — Añadir un agente Latinoamérica

Añade dos nuevas tools en `mcp_server.py`: `buscar_hoteles_latam` y `buscar_destinos_latam`.
Luego añade el agente `🌎 Latinoamérica` en `AGENT_TOOLS` y una cuarta pestaña en Gradio.

**Pista:** Los parámetros pueden ser iguales a los de Europa. Modifica la query de DuckDuckGo
para incluir términos como 'latinoamérica', 'sudamérica' o 'centroamérica' según corresponda.
Recuerda reiniciar el servidor con `servidor_proceso.terminate()` y volver a lanzarlo.


In [ ]:
# Reto 1 — Tu código aquí


### Reto 2 — Conectar Ollama como capa de inteligencia

En la arquitectura actual el usuario elige directamente qué tool llamar. En un sistema
real, un LLM decide. Modifica la aplicación para que el usuario escriba en lenguaje natural
("busca un hotel barato en Madrid para este fin de semana") y Ollama decida qué tool MCP llamar.

**Pista:** Instala `langchain-mcp-adapters` para convertir las tools MCP a formato LangChain.
Luego usa `ChatOllama` con `bind_tools()` para que el modelo elija automáticamente.


In [ ]:
# Reto 2 — Tu código aquí


### Reto 3 — Panel de auditoría de llamadas

Añade una pestaña extra en Gradio que muestre un historial de todas las llamadas realizadas:
agente, tool llamada, argumentos, timestamp y si fue permitida o denegada.

**Pista:** Usa una lista global `historial = []` y añade un registro en `ejecutar_con_control()`
antes de cada llamada. Usa `gr.Dataframe` para mostrar el historial en tiempo real.


In [ ]:
# Reto 3 — Tu código aquí


## 🌐 PARTE 11 — Publicar con Ngrok

Las siguientes celdas cierran la interfaz local y la publican con una URL
accesible desde cualquier dispositivo del mundo.

> **Requisito:** Token de Ngrok guardado en Colab Secrets con el nombre `NGROK_TOKEN`.
> Si no tienes token: regístrate gratis en [ngrok.com](https://ngrok.com) y créalo en **Your Authtoken**.


In [ ]:
# ============================================================
# CELDA 11.1 — Importar pyngrok
# ============================================================

from pyngrok import ngrok, conf  # Cliente Python de Ngrok para crear túneles HTTP
from google.colab import userdata  # Para leer secretos de Colab de forma segura

print('✅ pyngrok importado correctamente')


In [ ]:
# ============================================================
# CELDA 11.2 — Autenticar con el token de Ngrok
# ============================================================

# Leemos el token desde Colab Secrets (nunca hardcodeado en el notebook)
try:
    ngrok_token = userdata.get('NGROK_TOKEN')  # Clave guardada en los secretos de Colab
    conf.get_default().auth_token = ngrok_token  # Configuramos el token globalmente
    print('✅ Token de Ngrok configurado correctamente')
except Exception as e:
    print(f'❌ No se encontró NGROK_TOKEN en Colab Secrets: {e}')
    print('   Ve a 🔑 Secrets (icono de llave en el panel izquierdo) y añade NGROK_TOKEN')


In [ ]:
# ============================================================
# CELDA 11.3 — Cerrar interfaz local y publicar con Ngrok
# ============================================================

# Cerramos la interfaz local abierta en la Parte 8
demo.close()  # Libera el puerto 7860 antes de reabrir con Ngrok

# Cerramos túneles anteriores de Ngrok (si los hay) para evitar conflictos
ngrok.kill()

# Creamos un túnel HTTP del puerto 7860 (Gradio) al exterior
tunel_publico = ngrok.connect(7860)  # Devuelve un objeto con la URL pública

print(f'✅ Interfaz pública disponible en: {tunel_publico.public_url}')
print('   Comparte esta URL con tus compañeros para que prueben la demo en tiempo real')

# Relanzamos Gradio en localhost (Ngrok enruta el tráfico externo a este puerto)
demo.launch(
    server_name='0.0.0.0',  # Acepta conexiones de cualquier IP local
    server_port=7860,        # Puerto que Ngrok está tuneando
    share=False              # share=True es inestable; Ngrok es la solución profesional
)


In [ ]:
# ============================================================
# CELDA 11.4 — Verificar túneles activos
# ============================================================

# Listamos todos los túneles activos para confirmar que Ngrok está funcionando
tuneles = ngrok.get_tunnels()

if tuneles:
    print(f'✅ {len(tuneles)} túnel(es) activo(s):')
    for t in tuneles:
        print(f'   {t.name}: {t.public_url} → {t.config["addr"]}')
else:
    print('❌ No hay túneles activos. Vuelve a ejecutar la celda anterior.')


In [ ]:
# ============================================================
# CELDA LIMPIEZA — Liberar todos los recursos al terminar
# ============================================================

# 1. Cerrar el túnel Ngrok
ngrok.kill()
print('✅ Túnel Ngrok cerrado')

# 2. Cerrar la interfaz Gradio
demo.close()
print('✅ Interfaz Gradio cerrada')

# 3. Terminar el proceso del servidor MCP
servidor_proceso.terminate()      # Envía señal SIGTERM al proceso hijo
servidor_proceso.wait(timeout=5)  # Esperamos hasta 5s a que termine limpiamente
print(f'✅ Servidor MCP (PID {servidor_proceso.pid}) detenido')

print('\n✅ Todos los recursos liberados. Puedes cerrar el notebook.')


## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|----------|--------------|-------------------|-------------------|
| **Servidor MCP** | Las 6 tools están definidas y devuelven resultados reales de DuckDuckGo | Al menos 4 tools funcionan correctamente | El servidor arranca pero las tools fallan |
| **Control de acceso** | Los 3 agentes respetan sus restricciones; la denegación da mensaje claro | 2 de 3 agentes con restricciones correctas | AGENT_TOOLS definido pero sin verificación efectiva |
| **Interfaz Gradio** | 3 pestañas funcionales con secciones de hoteles y destinos en cada una | 2 pestañas funcionales | Gradio lanza pero los botones no responden |
| **Publicación Ngrok** | URL pública compartida y accesible desde otro dispositivo | Ngrok conecta pero la URL no es accesible | ngrok.connect() ejecutado sin verificación |
| **Reflexión** | Las 4 preguntas respondidas con ejemplos concretos y análisis propio | 3 preguntas respondidas con criterio | Menos de 3 respuestas o respuestas genéricas |
| **Reto opcional** | Al menos un reto completado y funcional | Reto iniciado con código relevante | Celda de reto vacía |
